In [1]:
# Base pyfantic

In [2]:
# ! pip install pydantic
# ! pip install pydantic-xml

# Атрибуты объекта в python - 
# '__annotations__',
# '__builtins__',
# '__call__',
# '__class__',
# '__closure__', '__code__',
# '__defaults__', '__delattr__', '__dict__',
# '__dir__', '__doc__',
# '__eq__', '__format__', '__ge__', '__get__',
# '__getattribute__',
# '__getstate__', '__globals__', '__gt__',
# '__hash__', '__init__', '__init_subclass__',
# '__kwdefaults__',
# '__le__', '__lt__', '__module__', '__name__',
# '__ne__', '__new__', '__qualname__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__type_params__']

In [3]:
from typing import Annotated, get_origin, get_args
from pydantic import BaseModel, Field, ValidationError
from pydantic.types import PositiveInt
from datetime import datetime

In [4]:
class User(BaseModel):
    username: str
    email: str


In [6]:
user = User(username="Saha", email='aa@ss.com')

In [2]:
PositiveIntStrict = Annotated[PositiveInt, Field(strict=True)]

In [6]:
print(PositiveIntStrict.__name__)
origin = get_origin(PositiveIntStrict)
args = get_args(PositiveIntStrict)
print(origin, args)

Annotated
<class 'typing.Annotated'> (<class 'int'>, Gt(gt=0), FieldInfo(annotation=NoneType, required=True, metadata=[Strict(strict=True)]))


In [14]:
class ItemStrict(BaseModel):
    try:
        qty: PositiveIntStrict
    except Exception as e:
        print(e)
        pass

In [15]:
dd = ItemStrict(qty=3)   # ✅ соответствует имени поля
print(dd.qty)

3


In [17]:
dd = ItemStrict(qty=10)   # ✅ соответствует имени поля
print(dd)

qty=10


In [22]:
print('1', dd.model_fields)                 # все поля модели
print('2', dd.model_fields["qty"].annotation)  # тип поля qty (будет PositiveIntStrict)
schema = dd.model_json_schema()
field_schema = schema["properties"]["qty"]

print('3', field_schema)
print('3', field_schema['title'])


1 {'qty': FieldInfo(annotation=int, required=True, metadata=[Gt(gt=0), Strict(strict=True)])}
2 <class 'int'>
3 {'exclusiveMinimum': 0, 'title': 'Qty', 'type': 'integer'}
3 Qty


/tmp/ipykernel_14922/499292686.py:1: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print('1', dd.model_fields)                 # все поля модели
/tmp/ipykernel_14922/499292686.py:2: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print('2', dd.model_fields["qty"].annotation)  # тип поля qty (будет PositiveIntStrict)


In [2]:
# Annotated types в Python — это способ добавить к обычному
#  типу дополнительные метаданные, которые не влияют на поведение программы
#  во время выполнения,

In [12]:
from pydantic import BaseModel, Field
from typing import Annotated

In [7]:
# Создаём переиспользуемый тип
Age = Annotated[int, Field(ge=-10, le=100)]


In [8]:
# Используем в модели
class User(BaseModel):
    age: Age

In [13]:
dd = User(age=12)
print(dd)
print(dd.age)

age=12
12


In [19]:
from typing import Annotated, Literal
from annotated_types import Gt
from pydantic import BaseModel

In [20]:
class Fruit(BaseModel):
  name: str  
  color: Literal['red', 'green']  
  weight: Annotated[float, Gt(0)]  
  bazam: dict[str, list[tuple[int, bool, float]]]

In [21]:
print(
  Fruit(
      name='Apple',
      color='red',
      weight=4.2,
      bazam={'foobar': [(1, True, 0.1)]},
  )
)

name='Apple' color='red' weight=4.2 bazam={'foobar': [(1, True, 0.1)]}


In [16]:
from pydantic import BaseModel, Field
from pydantic.dataclasses import dataclass

@dataclass
class Foo:
  bar: str
  baz: str = Field(init_var=True)
  qux: str = Field(kw_only=True)

class Model(BaseModel):
  foo: Foo


model = Model(foo=Foo('bar', baz='baz', qux='qux'))
print(model.model_dump())
print(model.foo.baz)

{'foo': {'bar': 'bar', 'qux': 'qux'}}
annotation=NoneType required=True init_var=True


In [8]:
# --- Pydantic V2 --- 
from typing import Annotated 
from annotated_types import Gt
from pydantic import BaseModel

# Означает: "целое число И оно должно быть больше 0" 
PositiveInt = Annotated[int, Gt(0)]

class UserV2(BaseModel):
    name: str 
    age: PositiveInt

user = UserV2(name="Alex", age=30)
data_v2 = user.model_dump()

print(data_v2)
print(type(user))
print(type(data_v2))

{'name': 'Alex', 'age': 30}
<class '__main__.UserV2'>
<class 'dict'>


In [10]:
from typing import List
from pydantic import TypeAdapter

# Описываем тип структуры 
UserListAdapter = TypeAdapter(List[UserV2]) 
raw_data = [ 
            {"name": "Alice", "age": 25}, 
            {"name": "Bob", "age": -5} # Здесь будет ошибка ValidationError 
            ] 
try: 
    valid_users = UserListAdapter.validate_python(raw_data) 
except Exception as e:
    print("❌ Error!")
    print(e.errors())

❌ Error!
[{'type': 'greater_than', 'loc': (1, 'age'), 'msg': 'Input should be greater than 0', 'input': -5, 'ctx': {'gt': 0}, 'url': 'https://errors.pydantic.dev/2.13/v/greater_than'}]


In [11]:
from uuid import UUID
from pydantic import TypeAdapter

uuid_adapter = TypeAdapter(UUID)
my_uuid = uuid_adapter.validate_python("f47ac10b-58cc-4372-a567-0e02b2c3d479") 
# Вернет объект UUID, либо упадет с ошибкой

In [13]:
# В) Генерация JSON Schema без создания класса: 
# Если вам нужна документация OpenAPI для сложного вложенного словаря,
# который вы не хотите оборачивать в BaseModel:

from typing import Dict

adapter = TypeAdapter(Dict[str, List[int]])
schema = adapter.json_schema()
# {'type': 'object', 'additionalProperties': {'type': 'array', 'items': {'type': 'integer'}}}
print(schema)

{'additionalProperties': {'items': {'type': 'integer'}, 'type': 'array'}, 'type': 'object'}


In [15]:
adapter_two = TypeAdapter(Dict[str, Dict[str, List[Dict[str, str]]]])
schema = adapter_two.json_schema()
print(schema)

{'additionalProperties': {'additionalProperties': {'items': {'additionalProperties': {'type': 'string'}, 'type': 'object'}, 'type': 'array'}, 'type': 'object'}, 'type': 'object'}


In [17]:
row1 = {'pole1': {'ur1_1': [{'ur_2_1': 'AA1',
                          'ur_2_2': "BB2"},
                         {'ur_2_3': 'AA3',
                          'ur_2_4': "BB4"}]}}
row1_a = adapter_two.validate_python(row1)
row1_a


{'pole1': {'ur1_1': [{'ur_2_1': 'AA1', 'ur_2_2': 'BB2'},
   {'ur_2_3': 'AA3', 'ur_2_4': 'BB4'}]}}